# 23 · Prioritize with several scored dimensions

## The problem

A backlog of feature requests (or sales leads, or bugs) needs ranking. Everyone agrees the ranking should weigh
**urgency, value, risk and strategic fit**, but they disagree on *how much* each one counts, and that changes every quarter.

If you ask a model *"what's the priority of this request, 1 to 10?"*, all of that is mashed into one opaque number.
You can't see why the confetti animation ranked above the bug fix, and when product says *"care less about risk this
quarter"*, the only lever you have is rewriting a prompt and hoping.

## Why Jev

Measure each dimension **separately**, with its own rubric, and keep the trade-off in **code**. A Jev `Score` returns
a calibrated position on a rubric you define. Four `Score` questions go in **one** `system_one` call (about 100 ms,
sub-cent per item), so scoring a whole backlog is cheap. The weights live in a Python dict that product can read,
argue about, and change without touching the model or re-scoring anything.

## What we'll build

```
feature request ──► Jev: 4 Scores in ONE call ──► normalise to 0..1 ──► weighted sum in code ──► ranked backlog
                     urgency  (0..3)                                   WEIGHTS = {urgency: +0.35,
                     value    (0..3)                                              value:   +0.35,
                     risk     (0..3)                                              risk:    -0.15,
                     fit      (0..3)                                              fit:     +0.25}
```

## What you'll learn
- How a `Score` rubric works, and why each dimension deserves its **own** question wording.
- The core design idea: **Jev measures, code decides**. Weights are policy; changing them re-ranks instantly from cached scores.
- How to explain any ranking line by line, and how to sanity-check dimensions for double-counting.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #17.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell loads `TYPESAFE_API_KEY` from the `.env` file at the repo root, creates the Jev client `jev`, and defines
`bar()`, a small helper that draws a 0..1 value as a text bar. We only need the **`Score`** question type here.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · One rubric per dimension

**How a `Score` works.** `criteria` is an **ordered rubric**: the first line is level 0, the last line is level 3.
Jev returns `.score`, a *continuous* position on that scale (2.4 means "between level 2 and level 3, closer to 2"),
plus `.confidence`. Each dimension below has a 4-level rubric, so scores run from 0 to 3.

Each dimension also gets its **own instructions**, saying exactly what to judge:

| Dimension | What it measures | Direction in the policy |
|---|---|---|
| `urgency` | time pressure: is anyone blocked or losing data now? | higher = do sooner |
| `value` | how much it helps users or revenue if shipped | higher = do sooner |
| `risk` | how much building it could break if we get it wrong | higher = **do later** |
| `fit` | match with this year's strategy | higher = do sooner |

> **Workshop note:** the first version used one template for all four, *"Rate this feature request on {dim}. Company
> strategy: ..."*, so the enterprise strategy leaked into urgency, value and risk too, and the four dimensions moved
> almost in lockstep. Now only `fit` sees the strategy, and `risk` says what kind of risk it means (risk of *building*
> it, not risk of *not* building it). The CSV-export bug, which loses customer data today, now scores as urgent as it should.

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [2]:
STRATEGY = "This year we focus on enterprise customers: SSO, audit logs, admin controls and reliability."

DIMENSIONS = {
    "urgency": ("How soon does this need to ship? Judge only time pressure: is anyone blocked or losing data right now?",
                ["Someday", "This quarter", "This month", "Blocking customers now"]),
    "value": ("If we ship this, how much value does it create for users or revenue?",
              ["Nice to have", "Helps a few users", "Helps many users", "Unlocks revenue or a major segment"]),
    "risk": ("How risky is building and shipping this change, i.e. how much could it break if we get it wrong?",
             ["Trivial change", "Contained change", "Touches core systems", "Could break billing, security or data"]),
    "fit": (f"How well does this fit the company strategy? Strategy: {STRATEGY}",
            ["Off-strategy", "Loosely related", "Aligned with the roadmap", "Core to this year's strategy"]),
}

SCORE_QUESTIONS = {dim: Score(instructions=ask, criteria=levels) for dim, (ask, levels) in DIMENSIONS.items()}
print("defined", len(SCORE_QUESTIONS), "Score questions:", list(SCORE_QUESTIONS))

defined 4 Score questions: ['urgency', 'value', 'risk', 'fit']


## Step 2 · Score the backlog once

`score_item()` sends one request with all four questions and divides each `.score` by its maximum (3) so every
dimension sits on the same **0..1** scale. Normalising matters: it makes the weights in Step 3 comparable.

We score the whole backlog **once** and cache the results in `SCORES`. Everything after this (re-weighting, the
sliders) reuses the cache, so changing the policy costs zero extra Jev calls.

This cell only **defines** the backlog and the function.

In [3]:
BACKLOG = [
    "SAML SSO for enterprise customers (3 deals waiting on it)",
    "Dark mode for the dashboard",
    "Fix: CSV export silently drops rows over 10k",
    "Rewrite the billing engine in Rust",
    "Audit log of admin actions, requested by our biggest customer's security team",
    "Animated confetti when a project is completed",
]


def score_item(item, client):
    """Four Scores in one call, each normalised to 0..1."""
    if "SCORE_QUESTIONS" not in globals():
        raise RuntimeError("SCORE_QUESTIONS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")
    r = client.system_one(state=item, questions=SCORE_QUESTIONS)
    return {dim: r.scores[dim].score / (len(levels) - 1) for dim, (_, levels) in DIMENSIONS.items()}

### Score all six items (six Jev calls)

In [4]:
SCORES = {item: score_item(item, jev) for item in BACKLOG}
for item, s in SCORES.items():
    print(f"{item[:55]:<55} " + "  ".join(f"{d[:4]} {v:.2f}" for d, v in s.items()))

SAML SSO for enterprise customers (3 deals waiting on i urge 0.89  valu 1.00  risk 0.99  fit 1.00
Dark mode for the dashboard                             urge 0.04  valu 0.36  risk 0.29  fit 0.15
Fix: CSV export silently drops rows over 10k            urge 0.91  valu 0.63  risk 0.75  fit 0.53
Rewrite the billing engine in Rust                      urge 0.02  valu 0.36  risk 1.00  fit 0.46
Audit log of admin actions, requested by our biggest cu urge 0.59  valu 0.77  risk 0.97  fit 1.00
Animated confetti when a project is completed           urge 0.01  valu 0.00  risk 0.05  fit 0.02


**What to notice**
- SAML SSO and the audit log score high on value and fit: they're literally the strategy. Dark mode and confetti score
  low almost everywhere.
- The CSV bug is about as urgent as SSO: customers are losing data today, even though it isn't "strategic".
- **SAML SSO also scores near the top for risk.** That's not a mistake. SSO replaces how enterprise users log in. If it
  breaks, people are locked out or, worse, let in wrongly. The top rubric level is literally *"Could break billing,
  security or data"*, and authentication is security. The audit log scores high for the same reason.
- The Rust rewrite of billing is the riskiest item of all, with low urgency: a classic "big, scary, can wait".

## Step 3 · The policy: explicit weights

The priority is a **weighted sum** of the four normalised scores:

`priority = 0.35·urgency + 0.35·value − 0.15·risk + 0.25·fit`

- Risk counts **against** priority, so its weight is negative.
- Its weight is deliberately **small**: risk is a *tie-breaker* between items of similar value, not a veto. A risky
  item that unlocks three enterprise deals should still be built, just carefully. If your team wants risk to veto,
  that's a different policy, and Step 3b shows what happens.
- Weights don't need to sum to 1. Only their relative size matters for the ranking.

To make the ranking **explainable**, `explain()` prints each dimension's *contribution* (weight × score). Anyone can
see exactly why an item landed where it did.

In [5]:
WEIGHTS = {"urgency": 0.35, "value": 0.35, "risk": -0.15, "fit": 0.25}


def priority(s, weights=WEIGHTS):
    return sum(weights[d] * s[d] for d in weights)


def explain(scores, weights=WEIGHTS):
    """Print the ranking with each dimension's contribution (weight x score)."""
    print(f"{'priority':>8}  " + "  ".join(f"{d:>8}" for d in weights) + "  item")
    for item in sorted(scores, key=lambda i: -priority(scores[i], weights)):
        s = scores[item]
        parts = "  ".join(f"{weights[d] * s[d]:+8.2f}" for d in weights)
        print(f"{priority(s, weights):+8.2f}  {parts}  {item[:48]}")


explain(SCORES)

priority   urgency     value      risk       fit  item
   +0.76     +0.31     +0.35     -0.15     +0.25  SAML SSO for enterprise customers (3 deals waiti
   +0.58     +0.21     +0.27     -0.15     +0.25  Audit log of admin actions, requested by our big
   +0.56     +0.32     +0.22     -0.11     +0.13  Fix: CSV export silently drops rows over 10k
   +0.14     +0.02     +0.12     -0.04     +0.04  Dark mode for the dashboard
   +0.10     +0.01     +0.13     -0.15     +0.11  Rewrite the billing engine in Rust
   +0.00     +0.00     +0.00     -0.01     +0.01  Animated confetti when a project is completed


**What to notice**
- SAML SSO and the audit log share the top two places, and the CSV fix is third. The ranking matches what a
  product lead would say for an enterprise-focused year.
- Read the SSO row: its high risk *costs* it about 0.15 points, but urgency, value and fit add far more. Risk lowers
  its score without changing its rank. That's the tie-breaker role at work.
- The Rust rewrite has some value but the biggest risk penalty and almost no urgency, so it sinks into the bottom
  half, even below dark mode.
- Dark mode and confetti sit at the bottom with priorities near zero: no dimension speaks up for them.

### Step 3b · Same scores, different policy

What if the CTO says *"this quarter, stability first"*? We triple the risk penalty, from −0.15 to −0.45. **No new
Jev calls:** we re-rank the cached `SCORES` with a different weights dict.

In [6]:
CAUTIOUS = {**WEIGHTS, "risk": -0.45}
explain(SCORES, CAUTIOUS)

priority   urgency     value      risk       fit  item
   +0.47     +0.31     +0.35     -0.44     +0.25  SAML SSO for enterprise customers (3 deals waiti
   +0.34     +0.32     +0.22     -0.34     +0.13  Fix: CSV export silently drops rows over 10k
   +0.29     +0.21     +0.27     -0.44     +0.25  Audit log of admin actions, requested by our big
   +0.05     +0.02     +0.12     -0.13     +0.04  Dark mode for the dashboard
   -0.01     +0.00     +0.00     -0.02     +0.01  Animated confetti when a project is completed
   -0.20     +0.01     +0.13     -0.45     +0.11  Rewrite the billing engine in Rust


**What to notice**
- Every priority drops, and the risky items drop the most. Compare the `risk` column with the table above.
- The CSV-export fix is the least risky of the top three, so it gains the most ground relative to SSO and the
  audit log. Depending on the exact scores it may now be level with them or ahead.
- The measurements didn't change at all. **Only the policy did.** That's the whole point of keeping weights in code.

## Step 4 · Sanity check: are we double-counting?

If two dimensions always move together, you're effectively counting the same thing twice and giving it double weight.
A correlation matrix over the scores shows it at a glance (1.0 = perfectly in step, 0 = unrelated).

In [7]:
import pandas as pd

df = pd.DataFrame(SCORES).T
print(df.round(2))
print("\ncorrelation between dimensions:")
print(df.corr().round(2))

                                                    urgency  value  risk   fit
SAML SSO for enterprise customers (3 deals wait...     0.89   1.00  0.99  1.00
Dark mode for the dashboard                            0.04   0.36  0.29  0.15
Fix: CSV export silently drops rows over 10k           0.91   0.63  0.75  0.53
Rewrite the billing engine in Rust                     0.02   0.36  1.00  0.46
Audit log of admin actions, requested by our bi...     0.59   0.77  0.97  1.00
Animated confetti when a project is completed          0.01   0.00  0.05  0.02

correlation between dimensions:
         urgency  value  risk   fit
urgency     1.00   0.85  0.56  0.75
value       0.85   1.00  0.79  0.93
risk        0.56   0.79  1.00  0.86
fit         0.75   0.93  0.86  1.00


**What to notice**
- Several correlations are high. With only **six** items, and most of them either "great on everything" (SSO, audit
  log) or "trivial on everything" (confetti, dark mode), that's expected. A tiny, polarised sample inflates correlation.
- Don't over-read a 6-row matrix. Run this check on a few hundred real backlog items. If two dimensions still correlate
  above about 0.9, merge them or drop one, or reword the rubrics so they measure different things (as the workshop note
  in Step 1 did).
- Not every pair is high: `urgency` and `risk` move together least. That's healthy, because risk is meant to be the
  counterweight.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a set of weight sliders and a form, wired to
the same `priority()` and `score_item()` functions. The scoring and policy logic above is untouched.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

Drag the weights: the ranking updates **instantly** from the cached `SCORES`, with no Jev calls. Try setting `risk` to
−0.6, or `fit` to 0, and watch the order change.

In [9]:
def board(urgency=0.35, value=0.35, risk=-0.15, fit=0.25):
    w = {"urgency": urgency, "value": value, "risk": risk, "fit": fit}
    rows = ""
    for rank, item in enumerate(sorted(SCORES, key=lambda i: -priority(SCORES[i], w)), 1):
        s = SCORES[item]
        dims = "".join(f'<span style="display:inline-block;width:70px;font-size:11px;color:{SLATE};">{d} {v:.2f}</span>'
                       for d, v in s.items())
        rows += (f'<div style="{CARD}padding:8px 12px;"><b>#{rank}</b> {badge(f"{priority(s, w):+.2f}", SAGE)} '
                 f'{item}<div>{dims}</div></div>')
    display(HTML(box(rows)))


sl = lambda v, lo=-1.0: widgets.FloatSlider(value=v, min=lo, max=1, step=0.05)
widgets.interact(board, urgency=sl(0.35), value=sl(0.35), risk=sl(-0.15), fit=sl(0.25));

interactive(children=(FloatSlider(value=0.35, description='urgency', max=1.0, min=-1.0, step=0.05), FloatSlide…

## Step 6 · Try it yourself

`item_html()` scores a **new** request (one Jev call) and shows its four dimensions plus its priority under the
default weights. Edit the text and press **Score** (needs a live kernel). Good ones to try:
- `Role-based permissions so admins can restrict who sees billing` (pre-filled) → high fit (admin controls are
  on-strategy) and moderate value, but risk near the top because it touches permissions and billing, so a middling priority.
- `Change the default font size from 14px to 15px` → low on everything, risk near 0.
- `Production database is down for all EU customers` → urgency near the top. Not a feature, but the rubric handles it.
- `Migrate all customer data to a new database cluster` → risk near the top, and the priority goes down for it.

In [10]:
def item_html(item):
    s = score_item(item, jev)
    meters = "".join(meter(d, v) for d, v in s.items())
    return box(f'<div style="{CARD}"><b>{item}</b> &nbsp;{badge(f"priority {priority(s):+.2f}", SAGE)}'
               f'<div style="margin-top:8px;">{meters}</div></div>')


form_app([("item", "Feature request", "Role-based permissions so admins can restrict who sees billing", 2)], item_html, "Score")

Output()

## Recap and exercises

**Key ideas**
1. **One `Score` per dimension**, each with its own rubric *and* its own wording. Vague or shared instructions blur the dimensions together.
2. **Jev measures, code decides.** The weights are policy: visible, debatable, and changeable without re-scoring.
3. **Explain every rank** with per-dimension contributions. A high risk score that doesn't change the rank is a feature, not a bug.
4. **Check for double-counting** with correlations, on enough data to mean something.

**Exercises**
- Make risk a **veto**: if `risk > 0.9` and `urgency < 0.5`, push the item to the bottom regardless of its sum. Which items move?
- Add a fifth dimension, `effort` (*"Hours" … "Quarters"*), with a negative weight. Does the Rust rewrite drop further?
- Change `STRATEGY` to *"This year we focus on self-serve consumers and delight"* and re-run Steps 2–3. Where do dark mode and confetti land now?
- Add 10 more realistic backlog items and re-run the correlation check. Do the correlations come down?